# Lekcija 11 - Protokol agent-agent (A2A)


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kaj je A2A protokol?

**Agent-to-Agent (A2A) protokol** je odprt standard, ki omogoča komunikacijo AI agentov,
odkrivanje drug drugega in sodelovanje — tudi če so zgrajeni na različnih okoljih ali gostovani
pri različnih storitvah.

Ključni pojmi:

- **Odkritje** – Agenti objavijo *Agentovo kartico*, ki opisuje njihove zmožnosti, kar olajša,
  da drugi agenti (ali orkestratorji) najdejo pravega strokovnjaka za nalogo.
- **Sporočanje** – Agenti izmenjujejo strukturirana sporočila preko skupnega protokola, tako da
  lahko zahtevo enega agenta razume in izvede drug agent ne glede na njegovo notranjo
  implementacijo.
- **Življenjski cikel naloge** – A2A definira stanja, kot so *poslano*, *v delu*, *zaključeno* in
  *neuspešno*, kar omogoča orkestratorju popoln pregled nad napredkom prenesene naloge.

V tej lekciji simuliramo sodelovanje v slogu A2A s povezovanjem treh specializiranih potovalnih agentov
v potek dela, kjer vsak agent prispeva svoje strokovno znanje in posreduje rezultate naslednjemu.


## Ustvarjanje specializiranih turističnih agentov


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Sodelovanje več agentov preko delovnega toka

Povežemo tri agente v zaporedni delovni tok, ki posnema prenos sporočil A2A:

1. **CurrencyExchangeAgent** prejme uporabniško zahtevo in pripravi navodila glede menjalnega tečaja.
2. **ActivityPlannerAgent** prejme obogateni kontekst in doda priporočila za aktivnosti.
3. **TravelManagerAgent** združi oba vhodna podatka v končno potovalno povzetek.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Razumevanje A2A v produkciji

V produkcijskem okolju protokol A2A omogoča močne scenarije med storitvami:

| Zmogljivost | Opis |
|---|---|
| **Interop med različnimi ogrodji** | Agent, zgrajen z enim ogrodjem, lahko delegira naloge agentu, zgrajenemu z katerim koli drugim A2A-kompatibilnim ogrodjem, kar omogoča pravo interoperabilnost med organizacijami. |
| **Meje storitev** | Agenti lahko delujejo v ločenih mikro storitvah, oblačnih regijah ali celo različnih organizacijah, hkrati pa brezhibno sodelujejo. |
| **Dinamično odkrivanje** | Orkestrator lahko med zagonom poizve registrator kartic agentov, da najde najbolj primernega specialista za dano podnalogo. |
| **Pretakanje in push obvestila** | A2A podpira Server-Sent Events (SSE) za posodobitve napredka v realnem času in push obvestila za dolgotrajne naloge. |

Delovni proces, ki smo ga ustvarili zgoraj, je poenostavljena, v procesu različica tega vzorca. V resnični
namestitvi bi vsak agent razkril HTTP končno točko, objavil kartico agenta in komuniciral
prek A2A JSON-RPC protokola.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
